# Finne rotasjonsvinkelen i 4D-STEM DPC

**Metoden.** Et felt fra en vortex (virvel) er enten rent *sirkulerende* (curl, uten divergens) eller rent *utoverrettet* (divergens, uten curl), aldri en blanding. Hvis vektorfeltet roteres med feil vinkel, blandes de to. Vi finner derfor vinkelen som gir best mulig skille mellom curl og divergens, og vurderer deretter hvor pålitelig den er.

**Innhold**
1. Last data og beregn DPC-signalet
2. Fjern descan
3. Verktøy for vinkelbestemmelse
4. Innstillinger
5. Kjør analysen og vurder påliteligheten
6. Roter feltet med funnet vinkel

In [ ]:
%matplotlib widget

## 1. Last data og beregn DPC-signalet

`hyperspy` leser datasettet, som lastes "lazy" slik at det ikke fyller minnet. `pyxem` brukes ikke direkte, men må importeres for at `get_direct_beam_position` skal være tilgjengelig. `get_direct_beam_position(method="center_of_mass")` gir en forskyvningsvektor for hver skannposisjon.

In [ ]:
import hyperspy.api as hs
import pyxem  
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

In [ ]:
s = hs.load("DPC_zspy/002.zspy", lazy=True)
s_bs = s.get_direct_beam_position(method="center_of_mass")
s_bs.compute()

### Komplekst DPC-signal

Vektoren (x, y) i hver skannposisjon lagres som ett komplekst tall `dpc = x + i·y`. Da blir en rotasjon av vektorfeltet en multiplikasjon med `exp(iθ)`, og en speiling blir `conj(dpc)`. Resultatet er en todimensjonal tabell med ett komplekst tall per skannposisjon.

In [ ]:
def to_complex(s_bs):
    d = np.asarray(s_bs.data if hasattr(s_bs, "data") else s_bs, dtype=float)
    return d[..., 0] + 1j * d[..., 1]


dpc_raw = to_complex(s_bs)

## 2. Fjern descan

Selv uten prøve kan strålen flytte seg på detektoren mens den skannes (*descan*). Det gir et lineært plan `a + b·x + c·y` oppå signalet. Et plan har konstant gradient, og det gir en falsk, konstant divergens og curl som forskyver vinkelen. Planet må derfor trekkes fra før vinkelen bestemmes.

`remove_descan` tilpasser planet med minste kvadrater, separat for x- og y-komponenten. Piksler som avviker mye fra planet (for eksempel selve vortexen) forkastes iterativt (*sigma-clipping*), så de ikke trekker planet bort fra bakgrunnsnivået. Funksjonen returnerer det korrigerte feltet og planet.

In [ ]:
def remove_descan(dpc, n_iter=5, clip=3.0):
    ny, nx = dpc.shape
    yy, xx = np.mgrid[:ny, :nx]
    A = np.column_stack([np.ones(dpc.size), xx.ravel(), yy.ravel()]).astype(float)
    valid = np.isfinite(dpc.ravel())
    plane = np.zeros(dpc.size, complex)
    for comp, factor in ((dpc.real, 1), (dpc.imag, 1j)):
        z = comp.ravel()
        used = valid.copy()
        for _ in range(n_iter):
            coef, *_ = np.linalg.lstsq(A[used], z[used], rcond=None)
            res = z - A @ coef # @ Utfører matrisemultiplikasjon
            new_used = valid & (np.abs(res) < clip * res[used].std())
            if new_used.sum() < 10 or (new_used == used).all():
                break
            used = new_used
        plane += (A @ coef) * factor
    plane = plane.reshape(dpc.shape)
    return dpc - plane, plane

In [ ]:
dpc, plane = remove_descan(dpc_raw)
print(f"rest-rms etter descan: {np.abs(dpc).std():.3f}")

## 3. Verktøy for vinkelbestemmelse

De følgende cellene definerer funksjonene som brukes i analysen. Selve analysen kjøres i del 5.

### Rotasjon, divergens og curl

- `rotate(dpc, theta)` dreier *komponentene* i vektorfeltet vinkelen `theta` (radianer). Selve bildet dreies ikke.
- `div_curl(v, sigma)` glatter feltet med et Gauss-filter (`sigma` er bredden i piksler, og demper støy) og beregner divergensen (hvor mye feltet strømmer ut av et punkt) og curl (hvor mye det sirkulerer rundt et punkt) fra gradientene. Piksel-y peker nedover i bildet, og curl er definert med det i bildekoordinater.

In [ ]:
def rotate(dpc, theta):
    return dpc * np.exp(1j * theta)


def div_curl(v, sigma=1.0):
    vx = ndi.gaussian_filter(v.real, sigma)
    vy = ndi.gaussian_filter(v.imag, sigma)
    dvx_dy, dvx_dx = np.gradient(vx)
    dvy_dy, dvy_dx = np.gradient(vy)
    return dvx_dx + dvy_dy, dvy_dx - dvx_dy

### Vinkelen i lukket form

La D og C være divergensen og curl for feltet uten rotasjon. Etter en rotasjon med vinkelen θ er
`div' = cos θ·D − sin θ·C` og `curl' = sin θ·D + cos θ·C`.
Summert over bildet blir differansen mellom curl-energien (Σcurl²) og divergensenergien (Σdiv²) en ren sinuskurve i 2θ. Toppen finnes direkte med `arctan2`, uten å prøve alle vinkler.

- Siden kurven er periodisk i 2θ, er vinkelen bare bestemt **modulo 180°**. Hvilket av de to svarene som er riktig, må avgjøres fra fysikken (se `flip180` i innstillingene).
- **R** er amplituden til kurven delt på total energi, og ligger mellom 0 og 1. R nær 1 betyr at feltet er rent curl (eller rent divergens) ved riktig vinkel. R nær 0 betyr at ingen vinkel er bedre enn andre, og da er vinkelen ikke bestemt.
- `want="curl"` søker vinkelen som gjør feltet rent sirkulerende (magnetisk-type kontrast). `want="div"` søker vinkelen som gjør det rent utoverrettet (rå Lorentz-avbøyning rundt en vortex). De to valgene gir svar som skiller seg med 90°.
- `estimate_rotation` returnerer vinkelen, R og scorekurven (for plotting). Med `erode > 0` fjernes så mange piksler langs bildekanten.

In [ ]:
def angle_from_sums(sdd, scc, sdc, want="curl"):
    A, B = scc - sdd, 2.0 * sdc
    if want == "div":
        A, B = -A, -B
    theta = (0.5 * np.arctan2(B, A)) % np.pi
    R = np.hypot(A, B) / (sdd + scc + 1e-30)
    return theta, R


def _erode(mask, n):
    mask = np.asarray(mask, bool)
    return ndi.binary_erosion(mask, iterations=n) if n > 0 else mask


def _sums(D, C, m):
    d, c = D[m], C[m]
    return np.sum(d * d), np.sum(c * c), np.sum(d * c)


def estimate_rotation(dpc, mask=None, sigma=1.5, want="curl", erode=0):
    m = _erode(np.ones(dpc.shape, bool) if mask is None else mask, erode)
    D, C = div_curl(dpc, sigma)
    sdd, scc, sdc = _sums(D, C, m)
    theta, R = angle_from_sums(sdd, scc, sdc, want)
    thetas = np.linspace(0, 2 * np.pi, 360, endpoint=False)
    A, B = scc - sdd, 2 * sdc
    if want == "div":
        A, B = -A, -B
    score = (A * np.cos(2 * thetas) + B * np.sin(2 * thetas)) / (sdd + scc + 1e-30)
    return theta, R, thetas, score

### Stabilitet mot valg av glatting

Svaret skal ikke avhenge av hvor mye feltet glattes eller hvor nær bildekanten pikslene er. `robust_rotation` gjentar estimatet for flere glattinger `sigmas` og flere kantbeskjæringer `erosions`, og kombinerer dem til én konsensusvinkel der hvert estimat er vektet med sin R.

Fordi vinkelen har periode 180°, brukes et *sirkulært* gjennomsnitt (`_circ`): 1° og 179° er nesten samme retning, og et vanlig gjennomsnitt ville gitt 90°. Spredningen i resultatet (`circ_std`) viser hvor mye estimatet varierer mellom innstillingene. Tabellen `table` har én rad per kombinasjon: `[sigma, erosjon, vinkel, R, antall piksler]`.

In [ ]:
def _circ(thetas, weights=None):
    w = np.ones_like(thetas) if weights is None else np.asarray(weights, float)
    z = np.sum(w * np.exp(2j * thetas)) / (np.sum(w) + 1e-30)
    mean = (0.5 * np.angle(z)) % np.pi
    rho = np.clip(np.abs(z), 1e-12, 1.0)
    std = 0.5 * np.sqrt(-2.0 * np.log(rho))
    return mean, std


def wrap_pi(d):
    return (d + np.pi / 2) % np.pi - np.pi / 2


def robust_rotation(dpc, mask=None, sigmas=(0.5, 1.0, 1.5, 2.0, 3.0), erosions=(0, 1, 2),
                    want="curl", min_pix=30):
    mask = np.ones(dpc.shape, bool) if mask is None else mask
    rows = []
    for s in sigmas:
        D, C = div_curl(dpc, s)
        for e in erosions:
            m = _erode(mask, e + int(round(s)))
            if m.sum() < min_pix:
                continue
            th, R = angle_from_sums(*_sums(D, C, m), want)
            rows.append((s, e, float(th), float(R), int(m.sum())))
    if not rows:
        raise ValueError("For få piksler igjen (se min_pix og erosions).")
    arr = np.array(rows)
    mean, std = _circ(arr[:, 2], arr[:, 3])
    return dict(theta=mean, circ_std=std, R_mean=float(arr[:, 3].mean()), table=arr)

### Usikkerhet med bootstrap

For å anslå hvor mye vinkelen ville variert med en annen prøve av samme data, trekkes bildet om igjen mange ganger (*bootstrap*) og vinkelen beregnes for hver omtrekning. Nabopiksler er korrelerte, så det trekkes hele blokker på `block × block` piksler med tilbakelegging, ikke enkeltpiksler. Resultatet har et 95 %-konfidensintervall `ci95` (avvik i grader fra snittet) og alle omtrukne vinkler `thetas`.

In [ ]:
def bootstrap_rotation(dpc, mask=None, sigma=1.5, want="curl", block=3, n_boot=2000, seed=0):
    mask = np.ones(dpc.shape, bool) if mask is None else mask
    D, C = div_curl(dpc, sigma)
    iy, ix = np.indices(D.shape)
    bid = (iy // block) * (ix.max() // block + 1) + (ix // block)
    m = np.asarray(mask, bool)
    _, inv = np.unique(bid[m], return_inverse=True)
    d, c = D[m], C[m]
    sdd = np.bincount(inv, d * d)
    scc = np.bincount(inv, c * c)
    sdc = np.bincount(inv, d * c)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(sdd), (n_boot, len(sdd)))
    th, R = angle_from_sums(sdd[idx].sum(1), scc[idx].sum(1), sdc[idx].sum(1), want)
    mean, std = _circ(th)
    dev = np.rad2deg(wrap_pi(th - mean))
    return dict(theta=mean, circ_std=std, ci95=np.percentile(dev, [2.5, 97.5]),
                thetas=th, R=R)

### Kontroll mot tilfeldighet og speiling

- `null_R` stokker pikslene tilfeldig, slik at all romlig struktur forsvinner, og beregner R. Fordelingen viser hvilken R man får av ren tilfeldighet. Den observerte R skal ligge godt over.
- `compare_hypotheses` beregner R både for `dpc` og for den speilede `conj(dpc)`. Hvis detektor og skann har ulik håndethet (speilvendt), passer den speilede versjonen bedre, og ingen ren rotasjon kan rette opp det.

In [ ]:
def null_R(dpc, mask=None, sigma=1.5, want="curl", n=300, seed=0):
    rng = np.random.default_rng(seed)
    m = np.ones(dpc.shape, bool) if mask is None else np.asarray(mask, bool)
    flat = dpc.ravel()
    out = np.empty(n)
    for i in range(n):
        shuf = flat[rng.permutation(flat.size)].reshape(dpc.shape)
        D, C = div_curl(shuf, sigma)
        out[i] = angle_from_sums(*_sums(D, C, m), want)[1]
    return out


def compare_hypotheses(dpc, mask=None, sigma=1.5, want="curl"):
    m = np.ones(dpc.shape, bool) if mask is None else np.asarray(mask, bool)
    res = {}
    for name, v in (("normal", dpc), ("speilet (conj)", np.conj(dpc))):
        D, C = div_curl(v, sigma)
        th, R = angle_from_sums(*_sums(D, C, m), want)
        res[name] = (float(th), float(R))
    return res

## 4. Innstillinger

Disse valgene kan ikke avgjøres av algoritmen.

| Variabel | Betydning |
|---|---|
| `want` | `"curl"`: feltet skal sirkulere rundt vortexen (tolkes som magnetisk kontrast). `"div"`: feltet skal peke utover/innover (rå Lorentz-avbøyning). Feil valg gir en vinkel som er 90° feil. |
| `sigma0` | Glatting (piksler) som brukes før gradientene beregnes. Større verdi demper støy, men viskes ut små strukturer. |
| `kant` | Antall piksler som utelates langs bildekanten, der gradient og glatting er upålitelige. |
| `flip180` | Vinkelen er bare bestemt modulo 180°. Sett til `True` hvis kontrasten du kjenner fra fysikken har motsatt fortegn. |

In [ ]:
want = "curl"
sigma0 = 1.5
kant = 2
flip180 = False

## 5. Kjør analysen og vurder påliteligheten

Cellen under beregner
- hovedestimatet `theta` og dets R,
- konsensusvinkelen `rob` over flere glattinger og kantbeskjæringer,
- bootstrap-fordelingen `boot`,
- nullfordelingen `null` for R,
- speiltesten `hyp`.

`mask_core` er hele bildet bortsett fra de ytterste `kant` pikslene.

In [ ]:
mask_core = _erode(np.ones(dpc.shape, bool), kant)
theta, R, thetas, score = estimate_rotation(dpc, mask_core, sigma=sigma0, want=want)
rob = robust_rotation(dpc, want=want)
boot = bootstrap_rotation(dpc, mask_core, sigma=sigma0, want=want)
null = null_R(dpc, mask_core, sigma=sigma0, want=want)
hyp = compare_hypotheses(dpc, mask_core, sigma=sigma0, want=want)

deg = np.rad2deg
ci_half = 0.5 * (boot["ci95"][1] - boot["ci95"][0])
R_null99 = np.percentile(null, 99)
R_mirror = hyp["speilet (conj)"][1]

### Vurdering

Seks tester avgjør om vinkelen kan stoles på. Hver skrives ut som `[OK]` eller `[FEIL]`.

1. R ligger over 99 %-persentilen av nullfordelingen (strukturen er ikke tilfeldig).
2. R er over 0,5 (feltet er tydelig curl eller divergens ved denne vinkelen).
3. Konsensusvinkelen varierer under 5° mellom glattinger og kantbeskjæringer.
4. Bootstrap-intervallet er smalere enn ±5°.
5. Hovedestimatet avviker mindre enn 5° fra konsensusvinkelen.
6. Den speilede hypotesen har R under halvparten av den normale.

Terskelene er fornuftige utgangspunkt, ikke faste grenser. Hvis speiltesten viser at `conj(dpc)` passer bedre, skrives det ut et tips om å teste `dpc = np.conj(dpc)`.

In [ ]:
checks = {
    f"R over null (R={R:.2f} > 99 %-persentil {R_null99:.2f})": R > R_null99,
    f"R høy nok (R={R:.2f} > 0.5)": R > 0.5,
    f"stabil over sigma/kant (std={deg(rob['circ_std']):.1f} grader < 5)": deg(rob["circ_std"]) < 5,
    f"bootstrap-CI smalt (±{ci_half:.1f} grader < 5)": ci_half < 5,
    f"hovedestimat likt konsensus (avvik {abs(deg(wrap_pi(theta - rob['theta']))):.1f} grader < 5)":
        abs(deg(wrap_pi(theta - rob["theta"]))) < 5,
    f"speilet hypotese dårligere (R_speil={R_mirror:.2f} < 0.5*R)": R_mirror < 0.5 * R,
}
print(f"theta (hoved)     = {deg(theta):6.1f} grader (mod 180)")
print(f"theta (konsensus) = {deg(rob['theta']):6.1f} ± {deg(rob['circ_std']):.1f} grader")
print(f"bootstrap 95 %    = ±{ci_half:.1f} grader")
print(f"R = {R:.2f}   null 99 %: {R_null99:.2f}   speilet R: {R_mirror:.2f}\n")
for k, v in checks.items():
    print(("[OK]   " if v else "[FEIL] ") + k)
n_ok = sum(checks.values())
print(f"\n{n_ok}/{len(checks)} tester bestått. " +
      ("Vinkelen ser pålitelig ut." if n_ok == len(checks) else
       "Vinkelen bør ikke brukes uten å undersøke hvilke tester som feilet."))
if hyp["speilet (conj)"][1] > hyp["normal"][1]:
    print("TIPS: conj(dpc) gir høyere R. Scann eller detektor er trolig speilvendt.")

### Diagnostikkplott

1. **Score mot θ**: sinuskurven fra del 3. Den røde stiplede linjen er valgt vinkel, og den stiplede kurven viser samme vinkel + 180°. En høy og tydelig topp betyr stor R.
2. **Stabilitet**: vinkelen for hver kombinasjon av glatting og kantbeskjæring, farget etter R. Punktene bør ligge samlet rundt den røde linjen (konsensusvinkelen).
3. **Bootstrap**: fordelingen av omtrukne vinkler rundt snittet. Smal fordeling betyr liten usikkerhet.
4. **Signifikans**: nullfordelingen av R. Den røde linjen (observert R) bør ligge langt til høyre for den.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
ax[0].plot(deg(thetas), score)
ax[0].axvline(deg(theta), c="r", ls="--")
ax[0].axvline(deg(theta) + 180, c="r", ls=":")
ax[0].set_xlabel("theta [grader]")
ax[0].set_ylabel("score")
ax[0].set_title(f"score (R={R:.2f})")

tab = rob["table"]
sc = ax[1].scatter(tab[:, 0], deg(tab[:, 2]), c=tab[:, 3], cmap="viridis", vmin=0, vmax=1)
plt.colorbar(sc, ax=ax[1], label="R")
ax[1].axhline(deg(rob["theta"]), c="r", ls="--")
ax[1].set_xlabel("sigma [px]")
ax[1].set_ylabel("theta [grader]")
ax[1].set_title("stabilitet")

ax[2].hist(deg(wrap_pi(boot["thetas"] - boot["theta"])), bins=40)
ax[2].set_xlabel("theta - snitt [grader]")
ax[2].set_title("bootstrap")

ax[3].hist(null, bins=30, alpha=0.7, label="null (stokket)")
ax[3].axvline(R, c="r", label="observert R")
ax[3].legend()
ax[3].set_xlabel("R")
ax[3].set_title("signifikans")
plt.tight_layout()

## 6. Roter feltet med funnet vinkel

Den endelige vinkelen `theta_final` er konsensusvinkelen fra del 5, pluss 180° hvis `flip180` er satt. `dpc_rot` er det roterte feltet, og `x_rot` og `y_rot` er komponentene.

Kontrollbildene viser divergens og curl av det roterte feltet. Med `want="curl"` skal curl-bildet ha tydelig struktur og divergensbildet være nær null, og omvendt for `want="div"`.

In [ ]:
theta_final = rob["theta"] + (np.pi if flip180 else 0.0)
print(f"theta_final = {deg(theta_final) % 360:.1f} grader")
dpc_rot = rotate(dpc, theta_final)
x_rot, y_rot = np.real(dpc_rot), np.imag(dpc_rot)

D, C = div_curl(dpc_rot, sigma0)
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(np.where(mask_core, D, np.nan), cmap="RdBu")
ax[0].set_title("divergens etter rotasjon")
ax[1].imshow(np.where(mask_core, C, np.nan), cmap="RdBu")
ax[1].set_title("curl etter rotasjon")